# Creating AI without such libraries as Torch or TensorFlow

In [2]:
import json

save_path = "hackernews_texts.json"
with open(save_path, "r") as f:
    hackernews_texts = json.load(f)

In [3]:
import re

def clean_text(text):
    text = re.sub(r"<[^>]+>", "", text)  # Remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9 .,!?]", "", text)  # Retain only valid characters
    return text

cleaned_texts = [clean_text(text) for text in hackernews_texts]
concatenated_text = "\n".join(cleaned_texts)

char_list = list()
for letter in concatenated_text:
    char_list.append(letter)

char_list[:10]
vocabulary = sorted(list(set(char_list)))

In [4]:
import numpy as np

stoi = {ch: number for number, ch in enumerate(vocabulary)}
encoded_text = [stoi[ch] for ch in concatenated_text]

tensored_data = np.array(encoded_text)
print(f"Encoded text length: {len(tensored_data)}")

n = int(0.9 * len(tensored_data))
train_data = tensored_data[:n]
val_data = tensored_data[n:]

print(f"Vocabulary length: {len(vocabulary)}")

Encoded text length: 61695
Vocabulary length: 68


In [5]:
def get_batch(split, block_size, batch_size):
    source = train_data if split == "train" else val_data

    ix = np.random.randint(0, len(source) - block_size - 1, size=batch_size)

    x = np.array([source[i:i+block_size] for i in ix])
    y = np.array([source[i + 1 : i + block_size + 1] for i in ix])
    return x, y

In [6]:
from NoTorchAI.Embedding import Embedding
from NoTorchAI.SGD import ABSGradient


class EmbeddingInitial:
    def __init__(self, vocab_size: int, d_model: int, block_size: int, gradient: ABSGradient):
        self.token_embedding = Embedding(vocab_size, d_model)
        self.position_embedding = Embedding(block_size, d_model)
        self.gradient = gradient

    def _change_weights(self) -> None:
        self.gradient.step(self.token_embedding)
        self.gradient.step(self.position_embedding)

    def forward(self, x):
        B, T = x.shape  

        token_emb = self.token_embedding.forward(x)
        pos_ids = np.arange(T)
        pos_emb = self.position_embedding.forward(pos_ids)

        x = token_emb + pos_emb  

        return x
    
    def backward(self, incoming_grad: np.ndarray) -> None:
        d_token = incoming_grad
        d_pos = np.sum(incoming_grad, axis=0)
        
        self.token_embedding.backward(d_token)
        self.position_embedding.backward(d_pos)

        self._change_weights()

In [7]:
from NoTorchAI.ActivationFunc import ReLu
from NoTorchAI.Layers.LinearLayer import Linear


class FeedForward:
    def __init__(self, d_model: int, gradient: ABSGradient):
        self.linear1 = Linear(d_model, 4 * d_model)
        self.relu = ReLu()
        self.linear2 = Linear(4 * d_model, d_model)

        self.gradient = gradient

    def _change_weights(self) -> None:
        self.gradient.step(self.linear1)
        self.gradient.step(self.linear2)

    def forward(self, x):
        x = self.linear1.forward(x)
        x = self.relu.forward(x)
        x = self.linear2.forward(x)
        return x
    
    def backward(self, incoming_grad: np.ndarray) -> np.ndarray:
        grad = self.linear2.backward(incoming_grad)
        grad = self.relu.backward(grad)
        grad = self.linear1.backward(grad)
        return grad

In [8]:
from NoTorchAI.SelfAttention import SelfAttention
from NoTorchAI.Layers.NormLayer import Normalization


class Block:
    def __init__(self, d_model: int, block_size: int, gradient: ABSGradient):
        super().__init__()
        self.linear1 = Normalization(d_model)
        self.attention = SelfAttention(d_model, gradient)
        self.linear2 = Normalization(d_model)
        self.ff = FeedForward(d_model, gradient)

        self.gradient = gradient

    def _change_weights(self) -> None:
        self.gradient.step(self.linear1)
        self.gradient.step(self.linear2)

    def forward(self, x: np.ndarray) ->  np.ndarray:
        x = self.linear1.forward(x)
        x = x + self.attention.forward(x)

        x = self.linear2.forward(x)
        x = x + self.ff.forward(x)
        
        return x
    
    def backward(self, incoming_grad: np.ndarray) -> np.ndarray:
        d_ff = incoming_grad
        d_x3_res = incoming_grad

        d_x3_ff = self.ff.backward(d_ff)
        d_x3 = d_x3_res + d_x3_ff

        d_x2 = self.linear2.backward(d_x3)

        d_attn = d_x2
        d_x1_res = d_x2

        d_x1_attn = self.attention.backward(d_attn)
        d_x1 = d_x1_res + d_x1_attn

        d_x = self.linear1.backward(d_x1)

        self._change_weights()
        return d_x

In [9]:
from NoTorchAI.CrossEntropy import CrossEntropy


class MiniGPT:
    def __init__(self, vocab_size: int, d_model: int, block_size: int, n_layers: int, gradient: ABSGradient):
        super().__init__()

        self.initial_embedding = EmbeddingInitial(vocab_size, d_model, block_size, gradient)

        self.blocks = [Block(d_model, block_size, gradient) for _ in range(n_layers)]

        self.linear = Normalization(d_model)
        self.head = Linear(d_model, vocab_size)
        self.cross_entropy = CrossEntropy()

        self.block_size = block_size

        self.gradient = gradient

    def _change_weights(self) -> None:
        self.gradient.step(self.linear)
        self.gradient.step(self.head) 

    def _execute_blocks_forward(self, x: np.ndarray) -> np.ndarray:
        for block in self.blocks:
            x = block.forward(x)
        return x
    
    def _execute_blocks_backward(self, grad: np.ndarray) -> np.ndarray:
        for block in self.blocks:
            grad = block.backward(grad)
        return grad 

    def forward(self, x: np.ndarray, targets: np.ndarray) -> tuple:
        B, T = x.shape

        x = self.initial_embedding.forward(x)

        x = self._execute_blocks_forward(x)

        x = self.linear.forward(x)

        logits = self.head.forward(x)
        loss = self.cross_entropy.forward(logits, targets)

        return logits, loss
    
    def backward(self):
        loss_grad = self.cross_entropy.backward()
        grad = self.head.backward(loss_grad)

        grad = self.linear.backward(grad)

        grad = self._execute_blocks_backward(grad)

        self.initial_embedding.backward(grad)


In [ ]:
from NoTorchAI.SGD import SGD


d_model = 128
vocabulary_size = len(vocabulary)
block_size = 64
batch_size = 32

sgd = SGD(lr=3e-4)
model = MiniGPT(vocab_size=vocabulary_size, d_model=128, block_size=64, n_layers=4, gradient=sgd)
xb, yb = get_batch("train", block_size, batch_size)

out = model.forward(xb, yb)
print(out[1])
model.backward()
out = model.forward(xb, yb)
print(out[1])

6.2520201519235385


In [11]:
# model = MiniGPT(vocabulary_size, d_model=128, block_size=64, n_layers=4)
# optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# for step in range(5000):
#     xb, yb = get_batch("train", block_size=64, batch_size=32)

#     logits, loss = model(xb, yb)

#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

#     if step % 500 == 0:
#         print("step:", step, "loss:", loss.item())